# BatchNorm作用验证实验
## 使用TensorBoard可视化分布统计

本实验将验证BatchNorm对训练稳定性的影响，通过hook获取每层的分布统计量（均值、方差、偏度、峰度等），并使用TensorBoard进行可视化。

## 1. 环境设置

In [ ]:
# 加载TensorBoard扩展
%load_ext tensorboard

# 安装必要的库
!pip install torch torchvision scipy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from scipy import stats
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 检查GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

## 2. 数据准备

In [ ]:
def prepare_data(batch_size=128):
    """准备CIFAR-10数据集"""
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(
        trainset, batch_size=batch_size, shuffle=True, num_workers=2)

    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"训练集大小: {len(trainset)}")
    print(f"测试集大小: {len(testset)}")
    
    return trainloader, testloader

# 准备数据
trainloader, testloader = prepare_data(batch_size=128)

## 3. 模型定义

In [ ]:
class SimpleCNN(nn.Module):
    """不带BatchNorm的CNN"""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.view(-1, 256 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class SimpleCNN_BN(nn.Module):
    """带BatchNorm的CNN"""
    def __init__(self, num_classes=10):
        super(SimpleCNN_BN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.bn_fc = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = x.view(-1, 256 * 4 * 4)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("模型定义完成")

## 4. 分布统计监控器

使用Hook机制捕获每层的输入，并计算统计量。

In [ ]:
class DistributionMonitor:
    """监控和记录层输入的分布统计量"""
    
    def __init__(self, writer, model_name):
        self.writer = writer
        self.model_name = model_name
        self.hooks = []
        self.epoch_stats = {}  # 存储每个epoch的统计量
        self.current_epoch = 0
        
    def compute_stats(self, tensor):
        """计算张量的统计量"""
        tensor_np = tensor.detach().cpu().numpy().flatten()
        
        if len(tensor_np) == 0:
            return None
            
        # 计算各种统计量
        mean = np.mean(tensor_np)
        std = np.std(tensor_np)
        variance = std ** 2
        
        # 高阶统计量
        skewness = stats.skew(tensor_np)
        kurtosis = stats.kurtosis(tensor_np)
        median = np.median(tensor_np)
        
        # 范围统计
        min_val = np.min(tensor_np)
        max_val = np.max(tensor_np)
        range_val = max_val - min_val
        
        # 分位数
        q25 = np.percentile(tensor_np, 25)
        q75 = np.percentile(tensor_np, 75)
        iqr = q75 - q25
        
        return {
            'mean': mean,
            'std': std,
            'variance': variance,
            'skewness': skewness,
            'kurtosis': kurtosis,
            'median': median,
            'min': min_val,
            'max': max_val,
            'range': range_val,
            'q25': q25,
            'q75': q75,
            'iqr': iqr
        }
    
    def create_hook(self, layer_name):
        """创建钩子函数"""
        def hook(module, input, output):
            if input is not None and len(input) > 0:
                input_tensor = input[0]
                stats = self.compute_stats(input_tensor)
                
                if stats is not None:
                    # 保存到epoch统计
                    if layer_name not in self.epoch_stats:
                        self.epoch_stats[layer_name] = []
                    self.epoch_stats[layer_name].append(stats)
        
        return hook
    
    def register_hooks(self, model, layer_names):
        """为指定层注册钩子"""
        self.remove_hooks()
        
        for name, module in model.named_modules():
            if name in layer_names:
                hook = self.create_hook(name)
                handle = module.register_forward_hook(hook)
                self.hooks.append((name, handle))
    
    def remove_hooks(self):
        """移除所有钩子"""
        for name, handle in self.hooks:
            handle.remove()
        self.hooks = []
    
    def log_epoch_stats(self, epoch):
        """将本epoch的统计量记录到TensorBoard"""
        self.current_epoch = epoch
        
        for layer_name, stats_list in self.epoch_stats.items():
            if len(stats_list) == 0:
                continue
                
            # 计算平均统计量
            avg_stats = {}
            for key in stats_list[0].keys():
                avg_stats[key] = np.mean([s[key] for s in stats_list])
            
            # 记录到TensorBoard
            for stat_name, stat_value in avg_stats.items():
                tag = f"{self.model_name}/{layer_name}/{stat_name}"
                self.writer.add_scalar(tag, stat_value, epoch)
        
        # 清空本epoch的统计
        self.epoch_stats = {}

print("监控器定义完成")

## 5. 训练函数

In [ ]:
def train_epoch(model, trainloader, criterion, optimizer, monitor, epoch):
    """训练一个epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(trainloader, desc=f'Epoch {epoch+1} [训练]', leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.3f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    train_loss = running_loss / len(trainloader)
    train_acc = 100. * correct / total
    
    return train_loss, train_acc


def test_epoch(model, testloader, criterion):
    """测试模型"""
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    test_loss = test_loss / len(testloader)
    test_acc = 100. * correct / total
    
    return test_loss, test_acc


def train_model(model, trainloader, testloader, monitor, layer_names,
                epochs=15, lr=0.001, model_name="model"):
    """完整训练流程"""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    
    print(f"\n{'='*60}")
    print(f"开始训练 {model_name}")
    print(f"{'='*60}")
    
    for epoch in range(epochs):
        # 注册钩子
        monitor.register_hooks(model, layer_names)
        
        # 训练
        train_loss, train_acc = train_epoch(
            model, trainloader, criterion, optimizer, monitor, epoch
        )
        
        # 记录epoch统计量到TensorBoard
        monitor.log_epoch_stats(epoch)
        
        # 移除钩子
        monitor.remove_hooks()
        
        # 测试
        test_loss, test_acc = test_epoch(model, testloader, criterion)
        
        # 记录到TensorBoard
        monitor.writer.add_scalar(f'{model_name}/Loss/train', train_loss, epoch)
        monitor.writer.add_scalar(f'{model_name}/Loss/test', test_loss, epoch)
        monitor.writer.add_scalar(f'{model_name}/Accuracy/train', train_acc, epoch)
        monitor.writer.add_scalar(f'{model_name}/Accuracy/test', test_acc, epoch)
        monitor.writer.add_scalar(f'{model_name}/Learning_Rate', 
                                 optimizer.param_groups[0]['lr'], epoch)
        
        print(f'Epoch {epoch+1}/{epochs}: '
              f'Train Loss: {train_loss:.3f}, Train Acc: {train_acc:.2f}%, '
              f'Test Loss: {test_loss:.3f}, Test Acc: {test_acc:.2f}%')
        
        scheduler.step()
    
    return model

print("训练函数定义完成")

## 6. 启动TensorBoard

In [ ]:
# 清理之前的日志
!rm -rf ./runs

# 启动TensorBoard
%tensorboard --logdir runs/batchnorm_experiment

## 7. 运行实验

In [ ]:
# 创建TensorBoard writers
writer_bn = SummaryWriter('runs/batchnorm_experiment/with_bn')
writer_no_bn = SummaryWriter('runs/batchnorm_experiment/without_bn')

# 创建模型
print("创建模型...")
model_with_bn = SimpleCNN_BN()
model_without_bn = SimpleCNN()

# 定义要监控的层
layers_bn = ['bn1', 'bn2', 'bn3', 'bn_fc']  # BatchNorm层
layers_no_bn = ['conv1', 'conv2', 'conv3', 'fc1']  # 对应的普通层

# 创建监控器
monitor_bn = DistributionMonitor(writer_bn, "WithBN")
monitor_no_bn = DistributionMonitor(writer_no_bn, "WithoutBN")

### 7.1 训练带BatchNorm的模型

In [ ]:
model_with_bn = train_model(
    model_with_bn, trainloader, testloader, monitor_bn,
    layers_bn, epochs=15, lr=0.001, model_name="带BatchNorm"
)

### 7.2 训练不带BatchNorm的模型

In [ ]:
model_without_bn = train_model(
    model_without_bn, trainloader, testloader, monitor_no_bn,
    layers_no_bn, epochs=15, lr=0.001, model_name="不带BatchNorm"
)

## 8. 关闭TensorBoard Writers

In [ ]:
writer_bn.close()
writer_no_bn.close()

print("\n" + "="*60)
print("实验完成！")
print("="*60)
print("\n请在上方的TensorBoard界面中查看结果")
print("\n在TensorBoard中可以查看:")
print("  - SCALARS tab: 训练/测试损失和准确率")
print("  - SCALARS tab: 各层的均值、方差、偏度、峰度等统计量")
print("  - 比较 'with_bn' 和 'without_bn' 两个实验的差异")
print("\n提示: 在TensorBoard左下角可以切换显示模式，选择多个指标进行对比")

## 9. 额外分析：直接可视化最后的分布

In [ ]:
import matplotlib.pyplot as plt

def visualize_final_activations(model_bn, model_no_bn, testloader, num_samples=10):
    """可视化最后一个batch的激活值分布"""
    # 获取一个batch
    dataiter = iter(testloader)
    images, labels = next(dataiter)
    images = images[:num_samples].to(device)
    
    # 收集激活值
    activations_bn = {}
    activations_no_bn = {}
    
    def save_activation(name, activations_dict):
        def hook(module, input, output):
            activations_dict[name] = output.detach().cpu().numpy().flatten()
        return hook
    
    # 注册钩子
    hook_bn = model_bn.bn1.register_forward_hook(save_activation('bn1', activations_bn))
    hook_no_bn = model_no_bn.conv1.register_forward_hook(save_activation('conv1', activations_no_bn))
    
    # 前向传播
    with torch.no_grad():
        _ = model_bn(images)
        _ = model_no_bn(images)
    
    # 移除钩子
    hook_bn.remove()
    hook_no_bn.remove()
    
    # 绘图
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 带BN的激活值
    axes[0].hist(activations_bn['bn1'], bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[0].set_title('带BatchNorm的第一层激活值分布')
    axes[0].set_xlabel('激活值')
    axes[0].set_ylabel('频数')
    axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='零点')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 不带BN的激活值
    axes[1].hist(activations_no_bn['conv1'], bins=50, alpha=0.7, color='orange', edgecolor='black')
    axes[1].set_title('不带BatchNorm的第一层激活值分布')
    axes[1].set_xlabel('激活值')
    axes[1].set_ylabel('频数')
    axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='零点')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印统计信息
    print("\n激活值统计对比:")
    print("="*60)
    print(f"{'指标':<15} {'带BatchNorm':<20} {'不带BatchNorm':<20}")
    print("-"*60)
    
    bn_acts = activations_bn['bn1']
    no_bn_acts = activations_no_bn['conv1']
    
    print(f"{'均值':<15} {np.mean(bn_acts):<20.4f} {np.mean(no_bn_acts):<20.4f}")
    print(f"{'标准差':<15} {np.std(bn_acts):<20.4f} {np.std(no_bn_acts):<20.4f}")
    print(f"{'偏度':<15} {stats.skew(bn_acts):<20.4f} {stats.skew(no_bn_acts):<20.4f}")
    print(f"{'峰度':<15} {stats.kurtosis(bn_acts):<20.4f} {stats.kurtosis(no_bn_acts):<20.4f}")
    print(f"{'最小值':<15} {np.min(bn_acts):<20.4f} {np.min(no_bn_acts):<20.4f}")
    print(f"{'最大值':<15} {np.max(bn_acts):<20.4f} {np.max(no_bn_acts):<20.4f}")

# 可视化
visualize_final_activations(model_with_bn, model_without_bn, testloader)

## 10. 总结

### 观察要点

在TensorBoard中，重点观察以下指标：

1. **均值（mean）**: BatchNorm应该使其接近0
2. **标准差（std）**: BatchNorm应该使其接近1
3. **偏度（skewness）**: BatchNorm应该使其接近0（对称分布）
4. **峰度（kurtosis）**: BatchNorm应该使其接近0（正态分布）
5. **训练稳定性**: 观察这些统计量在训练过程中的变化，BatchNorm应该使它们更稳定

### 预期结果

- 带BN的模型：分布统计量应该更稳定，接近标准正态分布
- 不带BN的模型：分布统计量可能会有较大波动，偏离正态分布
- 收敛速度：带BN的模型通常收敛更快
- 最终性能：带BN的模型通常有更好的泛化性能